# ETL-07 · Perfil 2021

- **Etapa:** ETL-07, Perfil de 2021
- **Responsável:** Samuel
- **Issue:** #9
- **Entrada:** `data/pessoas_*-2021.csv`
- **Saída:** `docs/perfil/perfil_2021.md` e `docs/perfil/corrompidos_2021.csv`

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd
import sys

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

DADOS  = RAIZ / "data"                      # CSVs brutos (estão no Git)

ENTRADA = DADOS      # pasta com os CSVs brutos
SAIDA   = RAIZ / "docs" / "perfil" / "perfil_2021.md"

sys.path.append(str(RAIZ / "src"))
from perfil import perfil, perfil_para_markdown

## 2.1 Teste de encoding

In [2]:
arquivos = [DADOS / f"pessoas_{mes:02d}-2021.csv" for mes in range(1, 13)]
for a in arquivos:
    try:
        a.read_bytes().decode("utf-8")
        print(a.name, "-> UTF-8 válido")
    except UnicodeDecodeError:
        print(a.name, "-> não é UTF-8 (provavelmente latin-1)")

pessoas_01-2021.csv -> não é UTF-8 (provavelmente latin-1)
pessoas_02-2021.csv -> não é UTF-8 (provavelmente latin-1)
pessoas_03-2021.csv -> não é UTF-8 (provavelmente latin-1)
pessoas_04-2021.csv -> não é UTF-8 (provavelmente latin-1)
pessoas_05-2021.csv -> não é UTF-8 (provavelmente latin-1)
pessoas_06-2021.csv -> não é UTF-8 (provavelmente latin-1)
pessoas_07-2021.csv -> não é UTF-8 (provavelmente latin-1)
pessoas_08-2021.csv -> não é UTF-8 (provavelmente latin-1)
pessoas_09-2021.csv -> não é UTF-8 (provavelmente latin-1)
pessoas_10-2021.csv -> não é UTF-8 (provavelmente latin-1)
pessoas_11-2021.csv -> não é UTF-8 (provavelmente latin-1)
pessoas_12-2021.csv -> não é UTF-8 (provavelmente latin-1)


## 2.2 Carregar

In [3]:
lista = []
for a in arquivos:
    d = pd.read_csv(a, encoding="latin-1", sep=";")
    d["arquivo_origem"] = a.name
    lista.append(d)

df = pd.concat(lista, ignore_index=True)
print("shape:", df.shape)   # esperado: (260886, 32)
df.head()

shape: (260886, 32)


,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,...,ano_mes_sinistro,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito,arquivo_origem
0,2240338,1979797,2874666.0,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,PASSAGEIRO,AUTOMOVEL,...,2021/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,pessoas_01-2021.csv
1,2240338,1979798,2874666.0,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,2021/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,pessoas_01-2021.csv
2,1447800,1284984,NaN,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,NaN,PEDESTRE,NaN,...,2021/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,pessoas_01-2021.csv
3,1783410,563279,2648403.0,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,2021/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,pessoas_01-2021.csv
4,1798856,687322,NaN,3547601,SANTA ROSA DE VITERBO,RIBEIRÃO PRETO,VIAS URBANAS,NaN,PASSAGEIRO,NaN,...,2021/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,pessoas_01-2021.csv


## 3. Diagnóstico ANTES

In [4]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'pct_nulo': (df.isna().mean() * 100).round(2),
        'n_distintos': df.nunique(),
    })

antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,128207
id_pessoa,int64,0.00,260886
id_veiculo,float64,21.21,178437
cod_ibge,int64,0.00,644
municipio,str,0.00,644
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,21.16,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.77,8


## 4. Cruzamento dos Pedestres

In [5]:
sem_veiculo = df[df["tipo_veiculo_vitima"].isna()]
print(sem_veiculo["tipo_de_vitima"].value_counts(dropna=False))
print("% sem veículo no total:", round(len(sem_veiculo) / len(df) * 100, 2))
print("% de pedestres entre os sem veículo:",
      round((sem_veiculo["tipo_de_vitima"] == "PEDESTRE").mean() * 100, 2))
print("% de pedestres sem veículo, no total:",
      round((sem_veiculo["tipo_de_vitima"] == "PEDESTRE").sum() / len(df) * 100, 2))

# sentido inverso 
ped = df[df["tipo_de_vitima"] == "PEDESTRE"]
print("pedestres:", len(ped), "| % deles sem veículo:",
      round(ped["tipo_veiculo_vitima"].isna().mean() * 100, 2))

tipo_de_vitima
NAO DISPONIVEL    32779
PEDESTRE          14468
PASSAGEIRO         6811
CONDUTOR           1134
Name: count, dtype: int64
% sem veículo no total: 21.16
% de pedestres entre os sem veículo: 26.21
% de pedestres sem veículo, no total: 5.55
pedestres: 14468 | % deles sem veículo: 100.0


## 5.1 Óbitos

In [6]:
# quantas linhas têm data de óbito preenchida (= quem morreu)
obitos = df['data_obito'].notna().sum()
print('óbitos:', obitos, '| % do total:', round(obitos / len(df) * 100, 2))

# as 7 colunas de óbito são nulas juntas?
# True  = em toda linha, as 7 estão TODAS vazias (não morreu) ou TODAS preenchidas (morreu)
# False = alguma linha tem óbito preenchido pela metade (dado quebrado)
cols_obito = ['data_obito', 'ano_obito', 'mes_obito', 'dia_obito',
              'ano_mes_obito', 'local_obito', 'tempo_sinistro_obito']
print((df[cols_obito].isna().all(axis=1) | df[cols_obito].notna().all(axis=1)).all())

óbitos: 4858 | % do total: 1.86
True


## 5.2 Extra - Nulos disfarçados

In [7]:
disfarcados = {"NAO DISPONIVEL", "NÃO DISPONÍVEL", "NAO INFORMADO", "NÃO INFORMADO"}
for col in df.select_dtypes(include=["object", "str"]).columns:
    n = df[col].dropna().astype(str).str.strip().str.upper().isin(disfarcados).sum()
    if n:
        print(col, n, f"({n / len(df) * 100:.2f}%)")

tipo_via 326 (0.12%)
tipo_veiculo_vitima 164 (0.06%)
tipo_de_vitima 37232 (14.27%)
modo_transporte_vitima 164 (0.06%)
sexo 3487 (1.34%)
gravidade_lesao 23548 (9.03%)
faixa_etaria_demografica 3475 (1.33%)
faixa_etaria_legal 3475 (1.33%)
profissao 49 (0.02%)
local_obito 26 (0.01%)
local_via 20842 (7.99%)


## 6. Valores corrompidos

In [8]:
# colunas de texto (o pandas novo guarda texto como 'str', por isso as duas opções)
colunas_texto = df.select_dtypes(include=['object', 'str']).columns

achados = []
for coluna in colunas_texto:
    # valores diferentes da coluna, sem os vazios
    valores = pd.Series(df[coluna].dropna().unique())

    # suspeitos: letra sozinha (acento perdido, ex.: "S O PAULO") ou "Ã" (acento mal lido)
    suspeitos = valores[valores.str.contains(r'\b[B-DF-NP-TV-Z]\b|Ã', regex=True)]

    # guarda cada suspeito com o nome da coluna onde apareceu
    for v in suspeitos:
        achados.append({'coluna': coluna, 'valor': v})

# transforma a lista numa tabela
corrompidos = pd.DataFrame(achados)
print(len(corrompidos), 'valores suspeitos')

# tira os valores com acento (estão certos) e fica só com os suspeitos de verdade
reais = corrompidos[corrompidos['valor'].map(str.isascii)]
print(len(reais), 'valores realmente suspeitos')
print(reais.to_string())

# salva a lista em arquivo
reais.to_csv(RAIZ / "docs" / "perfil" / "corrompidos_2021.csv", index=False)

14 valores suspeitos
8 valores realmente suspeitos
      coluna                   valor
0  municipio   SANTA BARBARA D OESTE
1  municipio       APARECIDA D OESTE
2  municipio         ESTRELA D OESTE
3  municipio        PALMEIRA D OESTE
4  municipio         GUARANI D OESTE
5  municipio      SANTA RITA D OESTE
6  municipio     SANTA CLARA D OESTE
7  municipio  SAO JOAO DO PAU D ALHO


## 7. Função perfil() e Geração de arquivo

In [9]:
p = perfil(df)

if SAIDA.exists():
    print("perfil_2021.md já existe, não sobrescrevi (apague para gerar de novo)")
else:
    _ = perfil_para_markdown(p, str(SAIDA), titulo="2021")

perfil_2021.md já existe, não sobrescrevi (apague para gerar de novo)


In [10]:

df["ano_sinistro"].unique()       # confirma que é só 2021

array([2021])

In [11]:
display(p["instrucao_por_mes"])       # P5: % de grau_de_instrucao por mês
display(p["texto_suspeito_resumo"])   # P4: espaços sobrando e outros motivos

,ano_mes,linhas,preenchidos,pct_preenchido
0,2021-01,19534,4418,22.62
1,2021-02,19684,4248,21.58
2,2021-03,18198,3819,20.99
3,2021-04,19586,4175,21.32
4,2021-05,23067,4986,21.62
5,2021-06,22192,4638,20.90
6,2021-07,24209,5064,20.92
7,2021-08,23852,5155,21.61
8,2021-09,21579,4762,22.07
9,2021-10,22772,5053,22.19


,coluna,motivo,valores_distintos,linhas
0,profissao,espaco_duplo,1,1
1,nacionalidade,espaco_nas_pontas,65,229731


## 8. Resumo

**O que foi feito:** perfil dos dados de 2021 (12 arquivos mensais `pessoas_*-2021.csv`), usando a função `perfil()`. Nenhum dado foi alterado; só foram feitos diagnósticos.

**Por quê:** as informações do perfil alimentam a comparação entre anos (ETL-12) e as etapas de limpeza e de tratamento de nulos e categorias (ETL-14 e ETL-16).

**Resultados:**
- 260.886 linhas × 31 colunas (mais `arquivo_origem`), com `id_pessoa` único e 128.207 sinistros (média de 2,03 pessoas por sinistro), todos de 2021.
- 644 municípios e 644 códigos IBGE.
- Nulos: `grau_de_instrucao` 78,29%; `id_veiculo` 21,21%; `tipo_veiculo_vitima` 21,16%; `modo_transporte_vitima` 20,77%; `nacionalidade` 11,94%; `profissao` 10,32%; `idade` 1,33%; `local_via` 1,17%.
- As 7 colunas de óbito têm 98,14% de nulos, sempre juntas.
- Óbitos: 4.858 (1,86% do total). As datas de óbito cobrem mais de um ano (`ano_obito` tem 2 valores distintos).
- Veículo nulo × tipo de vítima: 55.192 linhas (21,16%). Delas, 59,39% são "NAO DISPONIVEL", 26,21% pedestres, 12,34% passageiros e 2,05% condutores. No sentido inverso, 100% dos 14.468 pedestres estão sem veículo: a hipótese vale de pedestre para "sem veículo", mas não o contrário. Passageiros e condutores sem veículo (7.945 linhas) são registros incompletos. As três colunas do bloco de veículo têm % de nulos diferentes, então não ficam nulas exatamente nas mesmas linhas.
- `grau_de_instrucao` por mês: preenchido em cerca de 22% das linhas em todos os meses de 2021 (de 20,90% em junho a 22,62% em janeiro), sem diferença relevante entre eles. O campo é cerca de 78% nulo o ano inteiro, e não só fora de janeiro.
- Nulos disfarçados: "NAO DISPONIVEL" e "NAO INFORMADO" aparecem como texto, não como nulo, em `tipo_de_vitima` (37.232 linhas, 14,27%), `gravidade_lesao` (23.548, 9,03%) e `local_via` (20.842, 7,99%). Em menor escala aparecem em `sexo` (1,34%), nas duas faixas etárias (1,33%) e em `tipo_via` (0,12%).
- Texto: os 12 arquivos não são UTF-8 e foram lidos como `latin-1`; os acentos estão corretos. Os mesmos 8 municípios de 2022 ficaram com espaço no lugar do apóstrofo (ex.: "SANTA BARBARA D OESTE"). `municipio` vem sem acento e `regiao_administrativa` com acento. Todos os 65 valores distintos de `nacionalidade` têm espaços sobrando no fim (229.731 linhas) e `profissao` tem 1 valor com espaço duplo (1 linha). Nenhum valor com caractere inválido.

**Respostas às 10 perguntas padrão:** em `docs/perfil/perfil_2021.md`. Lista de valores suspeitos: `docs/perfil/corrompidos_2021.csv`.